# Ukázková úloha z PDF — rozbor

Oficiální zadání okruhu 1 (APR-I-II-3okruhy.pdf). Tohle je jediná úloha, o které víš jistě,
že je reprezentativní. Projdi ji celou, na časovku 60 minut.

**Postup:** přečti si zadání, zavři tenhle notebook a zkus to sám. Až potom se dívej na řešení.

## Zadání

> V následující funkci nalezněte syntaktické i sémantické chyby a opravte je.

Program rozšiřte o následující funkčnost:
- dodatečný **poziční parametr typu n-tice** určující dvojici znaků (otevírací a uzavírací závorka)
- **testování vstupní podmínky** (řetězec musí být neprázdný)
- funkce vracející **maximální úroveň zanoření** závorek
  (pokud nejsou správně uzávorkované, vyhazuje `ValueError`)

**Výstup:** opravený program + výsledky ladění (co funguje a co nikoliv a proč?)

### Původní kód — spusť ho NEJDŘÍV tak, jak je

In [ ]:
def test_of_parantheses(text: str) -> bool:
    """
       Testuje, zda jsou kulaté závorky správně uzávorkované.
      Příklad chybného uzávorkování: ")(())("
    """
    para_count = 0
    for c in text:
        if text == "(":
            para_count += 1
        elif text == ")":
            para_count -= 1
    return para_count == 0


# vyzkoušej na několika vstupech
for vzorek in ["(())", "()()", ")(", "(((", "", "úplně cokoliv"]:
    print(f"{vzorek!r:18} -> {test_of_parantheses(vzorek)}")

Co jsi viděl? **Všude `True`** — i pro `")("`, i pro `"úplně cokoliv"`.

Funkce nikdy nespadne a nikdy nevrátí `False`. To je klíčové pozorování: cyklus proběhne,
ale podmínka uvnitř nemůže nikdy platit.

---
## Tvoje řešení

Piš sem. Na řešení se dívej až potom.

In [ ]:
# 1) Oprava

In [ ]:
# 2) Rozšíření — n-tice závorek + kontrola vstupu

In [ ]:
# 3) Rozšíření — maximální zanoření

In [ ]:
# 4) Testy

---
---
# ŘEŠENÍ — nedívej se, dokud nemáš svoje

## Nalezené chyby

| # | Řádek | Typ | Popis | Oprava |
|---|-------|-----|-------|--------|
| 1 | 8, 10 | sémantická | `if text == "("` porovnává **celý řetězec** místo znaku `c` | `if c == "("` |
| 2 | 12 | sémantická | `para_count == 0` neodhalí `")("` — počet sedí, **pořadí ne** | test `para_count < 0` uvnitř cyklu |
| 3 | — | sémantická | Neošetřený prázdný vstup, zadání chce kontrolu vstupní podmínky | `if not text: raise ValueError` |

Ani jedna chyba není syntaktická — kód se spustí a doběhne. Chyba 1 je zákeřná tím, že
funkce vrací **pořád `True`**, takže na „hezkých" testech jako `"(())"` vypadá správně.

### Nejdůležitější postřeh z celé úlohy

**Naivní oprava (jen `c` místo `text`) je pořád špatně.** Ukážeme si to:

In [ ]:
def naivni_oprava(text):
    para_count = 0
    for c in text:
        if c == "(":            # opraveno: znak místo řetězce
            para_count += 1
        elif c == ")":
            para_count -= 1
    return para_count == 0


for vzorek in ["(())", "()()", ")(", "((("]:
    print(f"{vzorek!r:8} -> {naivni_oprava(vzorek)}")

Vidíš to? `")("` vrací **`True`**, přestože je to zjevně špatné uzávorkování —
a **PDF samo ho uvádí jako příklad chyby**. Jedna dolů, jedna nahoru, součet je nula.

Poučení do zkoušky: **testuj na vstupu, kde se správné a špatné řešení rozcházejí**,
ne na tom, který v zadání „vypadá dobře".

### Opravená verze

In [ ]:
def test_of_parantheses(text: str) -> bool:
    """Testuje, zda jsou kulaté závorky správně uzávorkované."""
    para_count = 0
    for c in text:
        if c == "(":                 # oprava 1: znak, ne celý řetězec
            para_count += 1
        elif c == ")":
            para_count -= 1
            if para_count < 0:       # oprava 2: zavírá dřív, než se otevřelo
                return False
    return para_count == 0


for vzorek in ["(())", "()()", ")(", "(((", "a(b)c"]:
    print(f"{vzorek!r:8} -> {test_of_parantheses(vzorek)}")

Ověření v hlavě:

| vstup | průběh počítadla | výsledek |
|---|---|---|
| `"(())"` | 1, 2, 1, 0 | `True` |
| `"()()"` | 1, 0, 1, 0 | `True` |
| `")("` | −1 → **hned `False`** | `False` |
| `"((("` | 1, 2, 3 → nekončí nulou | `False` |

### Rozšíření — všechny tři body ze zadání

In [ ]:
def test_of_parentheses(text: str, zavorky: tuple = ("(", ")")) -> bool:
    """
    Testuje, zda jsou závorky v textu správně uzávorkované.

    text     — neprázdný řetězec k otestování
    zavorky  — dvojice (otevírací, uzavírací) znak

    Vyhazuje ValueError, je-li text prázdný nebo dvojice neplatná.
    """
    if not isinstance(text, str):
        raise TypeError(f"očekávám řetězec, dostal jsem {type(text).__name__}")
    if not text:
        raise ValueError("vstupní řetězec musí být neprázdný")
    if len(zavorky) != 2 or zavorky[0] == zavorky[1]:
        raise ValueError(f"očekávám dvojici různých znaků, dostal jsem {zavorky!r}")

    otevirac, zavirac = zavorky        # rozbalení n-tice
    pocet = 0
    for znak in text:
        if znak == otevirac:
            pocet += 1
        elif znak == zavirac:
            pocet -= 1
            if pocet < 0:
                return False
    return pocet == 0


def max_zanoreni(text: str, zavorky: tuple = ("(", ")")) -> int:
    """
    Vrací maximální úroveň zanoření závorek.

    Vyhazuje ValueError, nejsou-li závorky správně uzávorkované.
    """
    if not test_of_parentheses(text, zavorky):
        raise ValueError(f"závorky v {text!r} nejsou správně uzávorkované")

    otevirac, zavirac = zavorky
    pocet = maximum = 0
    for znak in text:
        if znak == otevirac:
            pocet += 1
            maximum = max(maximum, pocet)    # zaznamenat vrchol
        elif znak == zavirac:
            pocet -= 1
    return maximum

**Proč `max_zanoreni` volá `test_of_parentheses` a nekontroluje si to samo:** zadání říká,
že při špatném uzávorkování má vyhodit `ValueError`, a kontrola už je hotová. Neduplikovat
logiku je věc, kterou u obhajoby ocení — a když se zeptají, řekni přesně tohle.

**Proč `zavorky` jako n-tice a ne dva parametry:** zadání to explicitně chce („dodatečný
poziční parametr typu n-tice"). Rozbalení `otevirac, zavirac = zavorky` je pak jednořádkové.

### Testy — včetně hraničních případů

In [ ]:
testy = [
    ("(())", True), ("()()", True), ("((()))", True),
    (")(", False), ("(((", False), (")))", False),
    ("a(b)c", True),      # znaky mimo dvojici se ignorují
    ("(", False), (")", False),
]

for vstup, ocekavano in testy:
    vysledek = test_of_parentheses(vstup)
    stav = "OK " if vysledek == ocekavano else "CHYBA"
    print(f"{stav} {vstup!r:10} -> {vysledek}")

In [ ]:
# vlastní dvojice závorek
print(test_of_parentheses("[a[b]]", ("[", "]")))     # True
print(test_of_parentheses("[[", ("[", "]")))         # False
print(test_of_parentheses("<x><y>", ("<", ">")))     # True

# maximální zanoření
for vzorek in ["((()))", "()()", "(())", "a(b(c)d)e"]:
    print(f"max_zanoreni({vzorek!r:12}) = {max_zanoreni(vzorek)}")

In [ ]:
# chybové stavy
for popis, volani in [
    ("prázdný řetězec",     lambda: test_of_parentheses("")),
    ("není řetězec",        lambda: test_of_parentheses(42)),
    ("stejné znaky",        lambda: test_of_parentheses("ab", ("x", "x"))),
    ("špatná délka n-tice", lambda: test_of_parentheses("ab", ("x",))),
    ("zanoření u chybného", lambda: max_zanoreni(")(")),
]:
    try:
        volani()
        print(f"{popis:22}: PROŠLO (nemělo!)")
    except (ValueError, TypeError) as e:
        print(f"{popis:22}: {type(e).__name__}: {e}")

---
## Výsledky ladění

*(Tohle je požadovaný výstup ze zadání. U zkoušky vyplň analogicky.)*

### Co funguje
- **Oprava:** `"(())"` a `"()()"` vrací `True`, `")("` a `"((("` vrací `False` — ověřeno ručně.
- **Klíčový test:** `")("` je vstup, na kterém naivní oprava selhává. Prochází → chyba 2 je opravdu opravená.
- **Rozšíření:** vlastní dvojice `("[", "]")` i `("<", ">")` funguje shodně s výchozí.
- **Zanoření:** `"((()))"` → 3, `"()()"` → 1, `"a(b(c)d)e"` → 2.
- **Kontrola vstupu:** prázdný řetězec i neplatná dvojice vyhodí `ValueError` podle zadání.
- **Hraniční:** jediná závorka `"("` i `")"` → `False`. Znaky mimo dvojici se ignorují.

### Co nefunguje / vědomá omezení
- **Nepodporuje víc typů závorek najednou.** `"([)]"` by prošlo jako správné, kdyby se testovaly
  odděleně. Na to je potřeba **zásobník** — při otevírací vlož, při zavírací zkontroluj vrchol.
- Znaky mimo zadanou dvojici se ignorují, což je zamýšlené (`"a(b)c"` je `True`).
- `test_of_parentheses` vrací `bool`, ale při neplatném **vstupu** vyhazuje výjimku —
  je to záměr: špatné uzávorkování je legitimní výsledek, prázdný vstup je chyba volajícího.

### Jak jsem testoval
- Krátké řetězce s ručně ověřeným výsledkem, ne generovaná data.
- Hraniční případy: prázdný vstup, jediná závorka, správný počet ve špatném pořadí.
- Chybové stavy: nesprávný typ, neplatná dvojice závorek.

### Složitost
- Čas $O(n)$ — každý znak se zpracuje jednou. Paměť $O(1)$ — drží se jen počítadlo.
- S podporou víc typů závorek by paměť byla $O(n)$ kvůli zásobníku.